In [1]:
import os

In [2]:
%pwd

'd:\\Coding\\Data science\\NLP-Project\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Coding\\Data science\\NLP-Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from textSummarizer.utils.common import read_yaml, create_directories
from textSummarizer.constants import *

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [14]:
import os
import requests
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [19]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            try:
                # Using requests instead of urllib
                response = requests.get(self.config.source_URL, stream=True)
                response.raise_for_status()  # Check for HTTP errors
                
                with open(self.config.local_data_file, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                
                logger.info(f"File downloaded successfully to {self.config.local_data_file}")
                
            except requests.exceptions.SSLError as e:
                logger.error(f"SSL Error occurred: {e}")
                # If you are behind a corporate proxy, try setting verify=False (use with caution)
                # or provide the path to your CA bundle: verify='/path/to/cert.pem'
                raise
            except Exception as e:
                logger.error(f"Error during download: {e}")
                raise
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")


    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [20]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-06-16 13:18:05,859: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-16 13:18:05,862: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-16 13:18:05,863: INFO: common: created directory at: artifacts]
[2026-06-16 13:18:05,864: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-16 13:21:16,554: INFO: 3622218058: File downloaded successfully to artifacts/data_ingestion/data.zip]


In [13]:
import ssl
print(ssl.OPENSSL_VERSION)

OpenSSL 3.5.7 9 Jun 2026
